# CATLA — Phase 6 LoRA fine-tuning (Kaggle, free T4/P100)

Same as `train_colab.ipynb` but for a Kaggle notebook (Settings -> Accelerator -> GPU T4 x2 or P100; Settings -> Internet -> On). Kaggle sessions get ~30 GPU-hrs/week and persist output under `/kaggle/working/` across a single session, but that directory is wiped between sessions unless committed as Kaggle Dataset output — so this notebook pushes checkpoints to the Hugging Face Hub (`--push_to_hub`) rather than relying on local persistence.

**Prerequisite**: Phase 5's `pipeline/push_to_hf.py` must already have been run (on your own machine, after `huggingface-cli login`).

**HF token**: add it via Kaggle's "Add-ons -> Secrets" as `HF_TOKEN`, not pasted into a cell.

In [ ]:
!nvidia-smi

In [ ]:
HF_DATASET_REPO = "CHANGE_ME/catla-bn-en-tweets"
HF_MODEL_REPO_PREFIX = "CHANGE_ME/catla"
GITHUB_REPO = "https://github.com/Shanjiv931/nlp_caslf.git"

ENGINES = {
    "indictrans2": "ai4bharat/indictrans2-en-indic-1B",
    "nllb": "facebook/nllb-200-distilled-600M",
    "banglat5": "csebuetnlp/banglat5",
}

In [ ]:
!git clone $GITHUB_REPO caslf
%cd caslf
!pip install -q -r requirements.txt
!pip install -q IndicTransToolkit
!pip uninstall -y torchao -q  # Kaggle's base image ships a torchao version too old for peft's LoRA
# dispatch check, which raises ImportError instead of skipping gracefully (hit this for real on
# 2026-08-05). We don't use torchao at all for plain LoRA, so removing it is the safe fix.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [ ]:
from datasets import load_dataset
import os

ds = load_dataset(HF_DATASET_REPO)
os.makedirs("data/processed", exist_ok=True)
for split, fname in [("train", "train.jsonl"), ("val", "val.jsonl"), ("test", "test.jsonl")]:
    ds[split].to_json(f"data/processed/{fname}", force_ascii=False)
print("data ready:", {k: len(v) for k, v in ds.items()})

In [ ]:
engine_key = "nllb"  # one of: indictrans2, nllb, banglat5
direction = "bn2en"  # or en2bn

model_name = ENGINES[engine_key]
output_dir = f"/kaggle/working/checkpoints/{engine_key}_{direction}"
hub_repo = f"{HF_MODEL_REPO_PREFIX}-{engine_key}-{direction}"

# Restrict to a single GPU. On a T4 x2 session, transformers' Trainer
# auto-wraps the model in naive DataParallel across both GPUs, but
# DataParallel funnels loss computation + gradient-gathering through GPU 0
# specifically, which can OOM that one GPU even with plenty of *combined*
# memory across both (hit this for real on 2026-08-05, mid-training).
# Single-GPU sidesteps the imbalance entirely rather than tuning around it.
!CUDA_VISIBLE_DEVICES=0 python pipeline/train.py \
  --model_name "$model_name" \
  --direction "$direction" \
  --output_dir "$output_dir" \
  --epochs 1 \
  --max_train_rows 150000 \
  --batch_size 8 \
  --grad_accum 4 \
  --save_steps 500 \
  --resume \
  --push_to_hub "$hub_repo"

If the ~30 GPU-hr/week quota runs out mid-engine, re-run this notebook next week. `--resume` tries a local checkpoint in `/kaggle/working/checkpoints/...` first (exact resume, but that directory is wiped when a Kaggle session ends, so this only helps if the *same* session is still warm); otherwise it falls back to pulling the LoRA weights from the Hub repo pushed via `--push_to_hub`. **The Hub fallback is an approximate resume** — the learned weights carry over, but the optimizer/learning-rate-schedule state does not (Trainer's automatic Hub push only uploads model files, not full training state), so the LR schedule restarts. Still much better than losing the trained weights and starting over.